# 🛰️ YC Startup Radar — Colab (all-in-one)

Run this notebook top to bottom in **Google Colab** to get:
1. a **final Excel** (`yc_radar.xlsx`) of YC startups (batches 2024–2026), and
2. an interactive **Streamlit dashboard** (via a temporary public link).

**Steps:** `Runtime → Run all`. Optionally add an `ANTHROPIC_API_KEY` (Colab **Secrets** 🔑) to fill in AI idea/risk summaries with **Claude Haiku 4.5** — paid but cheap (about **3–4 USD** for the whole set); without a key the AI columns show a placeholder and nothing is charged.

> Self-contained — no repo clone needed. It writes its own pipeline module and dashboard app into the Colab session.

## 1. Install dependencies

In [ ]:
!pip install -q httpx pandas pyarrow openpyxl anthropic streamlit

## 2. Write the pipeline module (`yc_radar_pipeline.py`)

In [ ]:
%%writefile yc_radar_pipeline.py
"""YC Startup Radar — self-contained pipeline module (Colab edition).

Consolidates the whole pipeline into one importable module so a single Colab
notebook can run everything without cloning the repo:

    fetch -> normalize -> enrich (investability + open links) -> score
          -> AI summaries (Haiku 4.5, cached) -> export (xlsx/parquet/csv)

Plus the Streamlit helpers (apply_filters) and personal annotations
(load/save/merge_user_data). Same logic as the tested `src/yc_radar` package.
"""

from __future__ import annotations

import json
import os
import re
import time
from collections.abc import Callable, Sequence
from pathlib import Path
from urllib.parse import quote_plus

import httpx
import pandas as pd
from openpyxl.utils import get_column_letter

# ======================================================================== fetch
COMPANIES_URL = "https://yc-oss.github.io/api/companies/all.json"
DEFAULT_CACHE_PATH = Path("data/raw/yc_companies.json")
_TIMEOUT_SECONDS = 60.0


def fetch_companies(
    *,
    force_refresh: bool = False,
    cache_path: Path = DEFAULT_CACHE_PATH,
    max_age_hours: float = 24.0,
    url: str = COMPANIES_URL,
) -> list[dict]:
    """Return YC company records, using a local cache when fresh."""
    cache_path = Path(cache_path)
    if not force_refresh and cache_path.exists():
        age = time.time() - cache_path.stat().st_mtime
        if age < max_age_hours * 3600:
            return json.loads(cache_path.read_text())
    try:
        response = httpx.get(url, timeout=_TIMEOUT_SECONDS)
        response.raise_for_status()
        records = response.json()
    except httpx.HTTPError as exc:
        raise RuntimeError(f"Failed to fetch YC companies from {url}: {exc}") from exc
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    cache_path.write_text(json.dumps(records))
    return records


# ==================================================================== normalize
DEFAULT_YEARS: tuple[int, ...] = (2024, 2025, 2026)
CORE_COLUMNS: tuple[str, ...] = (
    "name", "slug", "batch", "batch_year", "industry", "subindustry", "tags",
    "one_liner", "long_description", "status", "stage", "team_size", "location",
    "region", "is_hiring", "top_company", "nonprofit", "website", "yc_url",
    "launched_at",
)
_FULL_RE = re.compile(r"\b(20\d{2})\b")
_SHORT_RE = re.compile(r"^[WSFX](\d{2})$", re.IGNORECASE)


def parse_batch_year(batch: str | None) -> int | None:
    if not batch or not isinstance(batch, str):
        return None
    m = _FULL_RE.search(batch)
    if m:
        return int(m.group(1))
    m = _SHORT_RE.match(batch.strip())
    if m:
        return 2000 + int(m.group(1))
    return None


def _first_region(regions: object) -> str:
    if isinstance(regions, list) and regions:
        return str(regions[0])
    return ""


def normalize(records: list[dict], *, years: tuple[int, ...] = DEFAULT_YEARS) -> pd.DataFrame:
    if not records:
        return pd.DataFrame(columns=list(CORE_COLUMNS))
    df = pd.DataFrame(records)
    df["batch_year"] = df.get("batch").map(parse_batch_year).astype("Int64")
    df["is_hiring"] = df.get("isHiring", False).astype("boolean").fillna(False).astype(bool)
    df["yc_url"] = df.get("url", "")
    df["location"] = df.get("all_locations", "").fillna("")
    df["region"] = df.get("regions").map(_first_region)
    df["team_size"] = pd.to_numeric(df.get("team_size"), errors="coerce").astype("Int64")
    for col in ("name", "slug", "batch", "industry", "subindustry", "one_liner",
                "long_description", "status", "stage", "website"):
        if col not in df.columns:
            df[col] = ""
    if "tags" not in df.columns:
        df["tags"] = [[] for _ in range(len(df))]
    for flag in ("top_company", "nonprofit"):
        df[flag] = df.get(flag, False).astype("boolean").fillna(False).astype(bool)
    if "launched_at" not in df.columns:
        df["launched_at"] = pd.NA
    df = df[df["batch_year"].isin(set(years))].copy()
    df = df.drop_duplicates(subset="slug", keep="first")
    return df[list(CORE_COLUMNS)].reset_index(drop=True)


# ======================================================================= enrich
INVESTABILITY: dict[str, str] = {
    "Public": "Public — buyable on the open market",
    "Acquired": "Acquired — not directly investable",
    "Active": "Private — accredited / SPV / secondary only",
    "Inactive": "Inactive — not investable",
}
INVESTABILITY_UNKNOWN = "Unknown"

LINK_BUILDERS: dict[str, str] = {
    "news_url": "https://news.google.com/search?q={q}",
    "producthunt_url": "https://www.producthunt.com/search?q={q}",
    "hn_url": "https://hn.algolia.com/?q={q}",
    "github_url": "https://github.com/search?q={q}&type=repositories",
    "wikipedia_url": "https://en.wikipedia.org/w/index.php?search={q}",
}


def add_investability(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["investability"] = out["status"].map(INVESTABILITY).fillna(INVESTABILITY_UNKNOWN)
    return out


def add_links(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    names = out["name"].fillna("") if "name" in out.columns else pd.Series([""] * len(out))
    for col, template in LINK_BUILDERS.items():
        out[col] = [template.format(q=quote_plus(str(n))) if str(n).strip() else "" for n in names]
    return out


# ======================================================================== score
DEFAULT_WEIGHTS: dict[str, float] = {
    "top_company": 3.0, "recency": 2.0, "hiring": 1.0,
    "team": 1.0, "description": 0.5, "tags": 0.5,
}
_MIN_YEAR = 2023


def _tag_count(tags: object) -> int:
    if isinstance(tags, list):
        return len(tags)
    if isinstance(tags, str) and tags.strip():
        return len([t for t in tags.split(",") if t.strip()])
    return 0


def _components(row: pd.Series) -> dict[str, float]:
    year = row.get("batch_year")
    team = row.get("team_size")
    desc = row.get("long_description") or ""
    return {
        "top_company": 1.0 if bool(row.get("top_company")) else 0.0,
        "recency": (min(max((int(year) - _MIN_YEAR) / 3.0, 0.0), 1.0) if pd.notna(year) else 0.0),
        "hiring": 1.0 if bool(row.get("is_hiring")) else 0.0,
        "team": (min(int(team), 50) / 50.0 if pd.notna(team) else 0.0),
        "description": 1.0 if len(str(desc)) >= 40 else 0.0,
        "tags": min(_tag_count(row.get("tags")), 3) / 3.0,
    }


def score(df: pd.DataFrame, *, weights: dict[str, float] | None = None) -> pd.DataFrame:
    w = {**DEFAULT_WEIGHTS, **(weights or {})}
    total_w = sum(w.values()) or 1.0

    def _row_score(row: pd.Series) -> float:
        comps = _components(row)
        raw = sum(w.get(k, 0.0) * v for k, v in comps.items())
        return round(100.0 * raw / total_w, 1)

    out = df.copy()
    out["score"] = out.apply(_row_score, axis=1) if len(out) else []
    return out


# =========================================================================== ai
AI_MODEL = "claude-haiku-4-5"
AI_CACHE_PATH = Path("data/processed/ai_cache.json")
AI_DISABLED = "AI summary disabled (set ANTHROPIC_API_KEY to enable)"
MAX_DESC_CHARS = 1500  # cost control: cap description input tokens
_AI_INSTRUCTION = (
    "You are a venture analyst. Given a startup's name and description, write a "
    "clear, information-dense brief for an investor scanning many companies. Return "
    "JSON with two fields: 'summary' (2-3 sentences: what the company does, who it "
    "is for, and what makes it distinctive or notable) and 'risks' (1-2 concrete "
    "things to check before investing - e.g. market size, competition, moat, "
    "regulatory, or execution risk). Be specific; base it only on the provided "
    "text and do not invent facts, numbers, or financials."
)
_AI_SCHEMA = {
    "type": "object",
    "properties": {"summary": {"type": "string"}, "risks": {"type": "string"}},
    "required": ["summary", "risks"],
    "additionalProperties": False,
}
Summarizer = Callable[[list[dict], str], dict[str, dict[str, str]]]


def _ai_load_cache(cache_path: Path) -> dict[str, dict[str, str]]:
    cache_path = Path(cache_path)
    return json.loads(cache_path.read_text()) if cache_path.exists() else {}


def _ai_save_cache(cache_path: Path, cache: dict) -> None:
    cache_path = Path(cache_path)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = cache_path.with_suffix(cache_path.suffix + ".tmp")
    tmp.write_text(json.dumps(cache, indent=2, ensure_ascii=False))
    tmp.replace(cache_path)


def _ai_records_for(df: pd.DataFrame, slugs: list[str]) -> list[dict]:
    sub = df[df["slug"].isin(slugs)]
    return [
        {"slug": r["slug"], "name": r.get("name", ""),
         "one_liner": r.get("one_liner", ""), "long_description": r.get("long_description", "")}
        for _, r in sub.iterrows()
    ]


def _ai_batch_summarize(records: list[dict], model: str, api_key: str) -> dict[str, dict[str, str]]:
    import anthropic
    from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
    from anthropic.types.messages.batch_create_params import Request

    client = anthropic.Anthropic(api_key=api_key)
    system = [{"type": "text", "text": _AI_INSTRUCTION, "cache_control": {"type": "ephemeral"}}]
    requests = [
        Request(
            custom_id=rec["slug"],
            params=MessageCreateParamsNonStreaming(
                model=model, max_tokens=500, system=system,
                messages=[{"role": "user", "content": _ai_user_prompt(rec)}],
                output_config={"format": {"type": "json_schema", "schema": _AI_SCHEMA}},
            ),
        )
        for rec in records
    ]
    batch = client.messages.batches.create(requests=requests)
    while client.messages.batches.retrieve(batch.id).processing_status != "ended":
        time.sleep(10)
    out: dict[str, dict[str, str]] = {}
    for res in client.messages.batches.results(batch.id):
        if res.result.type != "succeeded":
            continue
        text = next((b.text for b in res.result.message.content if b.type == "text"), "{}")
        data = json.loads(text)
        out[res.custom_id] = {"ai_summary": data.get("summary", ""),
                              "ai_risk_notes": data.get("risks", "")}
    return out


GROQ_DEFAULT_MODEL = "llama-3.1-8b-instant"


def _ai_user_prompt(rec: dict) -> str:
    _d = str(rec.get("long_description", ""))[:MAX_DESC_CHARS]
    return (f"Company: {rec['name']}\nOne-liner: {rec['one_liner']}\n"
            f"Description: {_d}")


def _groq_one(client, model, rec, *, max_retries=5):
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "system", "content": _AI_INSTRUCTION},
                          {"role": "user", "content": _ai_user_prompt(rec)}],
                response_format={"type": "json_object"}, max_tokens=500, temperature=0.3,
            )
            data = json.loads(resp.choices[0].message.content)
            return {"ai_summary": str(data.get("summary", "")).strip(),
                    "ai_risk_notes": str(data.get("risks", "")).strip()}
        except Exception as exc:
            last_err = exc
            time.sleep(min(2 ** attempt, 30))
    return {"ai_summary": f"(Groq summary failed: {last_err})", "ai_risk_notes": ""}


def make_groq_summarizer(api_key=None, *, model=GROQ_DEFAULT_MODEL, cache_path=None,
                         sleep=1.0, max_retries=5, client=None):
    """Free-tier **Groq** summarizer (no Claude key). Sequential, paced, resumable."""
    if client is None:
        from groq import Groq
        client = Groq(api_key=api_key or os.environ.get("GROQ_API_KEY"))

    def summarizer(records, _model=""):
        persisted = _ai_load_cache(cache_path) if cache_path else {}
        out = {}
        for rec in records:
            res = _groq_one(client, model, rec, max_retries=max_retries)
            out[rec["slug"]] = res
            if cache_path:
                persisted[rec["slug"]] = res
                _ai_save_cache(cache_path, persisted)
            if sleep:
                time.sleep(sleep)
        return out

    return summarizer


def _claude_one(client, model, rec, *, max_tokens=300, max_retries=5):
    system = [{"type": "text", "text": _AI_INSTRUCTION, "cache_control": {"type": "ephemeral"}}]
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.messages.create(
                model=model, max_tokens=max_tokens, system=system,
                messages=[{"role": "user", "content": _ai_user_prompt(rec)}],
                output_config={"format": {"type": "json_schema", "schema": _AI_SCHEMA}},
            )
            text = next((b.text for b in resp.content if b.type == "text"), "{}")
            data = json.loads(text)
            return {"ai_summary": str(data.get("summary", "")).strip(),
                    "ai_risk_notes": str(data.get("risks", "")).strip()}
        except Exception as exc:
            last_err = exc
            time.sleep(min(2 ** attempt, 30))
    return {"ai_summary": f"(Claude summary failed: {last_err})", "ai_risk_notes": ""}


def make_claude_summarizer(api_key=None, *, model=AI_MODEL, cache_path=None,
                           max_tokens=500, sleep=0.0, max_retries=5,
                           progress_every=50, client=None):
    """Synchronous **Claude** summarizer (paid, cheap): truncated input, short output,
    resumable cache, progress prints."""
    if client is None:
        import anthropic
        client = anthropic.Anthropic(api_key=api_key or os.environ.get("ANTHROPIC_API_KEY"))

    def summarizer(records, _model=""):
        persisted = _ai_load_cache(cache_path) if cache_path else {}
        out = {}
        total = len(records)
        for i, rec in enumerate(records, 1):
            out[rec["slug"]] = _claude_one(client, model, rec, max_tokens=max_tokens,
                                           max_retries=max_retries)
            if cache_path:
                persisted[rec["slug"]] = out[rec["slug"]]
                _ai_save_cache(cache_path, persisted)
            if progress_every and (i % progress_every == 0 or i == total):
                print(f"  AI summaries: {i}/{total} companies…", flush=True)
            if sleep:
                time.sleep(sleep)
        return out

    return summarizer


def add_ai_summaries(
    df: pd.DataFrame, *, cache_path: Path = AI_CACHE_PATH, model: str = AI_MODEL,
    api_key: str | None = None, summarizer: Summarizer | None = None,
) -> pd.DataFrame:
    cache = _ai_load_cache(cache_path)
    slugs = df["slug"].tolist()
    missing = [s for s in slugs if s not in cache]
    key = api_key or os.environ.get("ANTHROPIC_API_KEY")
    if missing:
        if summarizer is not None:
            cache.update(summarizer(_ai_records_for(df, missing), model))
            _ai_save_cache(cache_path, cache)
        elif key:
            cache.update(_ai_batch_summarize(_ai_records_for(df, missing), model, key))
            _ai_save_cache(cache_path, cache)
    out = df.copy()
    out["ai_summary"] = [cache.get(s, {}).get("ai_summary", AI_DISABLED) for s in slugs]
    out["ai_risk_notes"] = [cache.get(s, {}).get("ai_risk_notes", AI_DISABLED) for s in slugs]
    return out


# =================================================================== user_data
USER_COLUMNS = ("slug", "my_rating", "watchlist", "my_notes")
USER_DATA_PATH = Path("data/user_data.csv")


def load_user_data(path: Path = USER_DATA_PATH) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        return pd.DataFrame(columns=list(USER_COLUMNS))
    df = pd.read_csv(path)
    for col in USER_COLUMNS:
        if col not in df.columns:
            df[col] = pd.NA
    return df[list(USER_COLUMNS)]


def save_user_data(rows: list[dict] | pd.DataFrame, path: Path = USER_DATA_PATH) -> None:
    path = Path(path)
    df = pd.DataFrame(rows) if not isinstance(rows, pd.DataFrame) else rows.copy()
    for col in USER_COLUMNS:
        if col not in df.columns:
            df[col] = pd.NA
    df = df[list(USER_COLUMNS)]
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    tmp.replace(path)


def merge_user_data(df: pd.DataFrame, path: Path = USER_DATA_PATH) -> pd.DataFrame:
    user = load_user_data(path)
    out = df.merge(user, on="slug", how="left")
    out["watchlist"] = out["watchlist"].astype("boolean").fillna(False).astype(bool)
    out["my_notes"] = out["my_notes"].fillna("").astype(str)
    out["my_rating"] = pd.to_numeric(out["my_rating"], errors="coerce").astype("Int64")
    return out


# ====================================================================== filters
_SEARCH_FIELDS = ("name", "one_liner", "long_description", "tags")


def _row_text(row: pd.Series) -> str:
    parts = []
    for field in _SEARCH_FIELDS:
        val = row.get(field)
        if isinstance(val, list):
            parts.append(" ".join(map(str, val)))
        elif pd.notna(val):
            parts.append(str(val))
    return " ".join(parts).lower()


def apply_filters(
    df: pd.DataFrame, *, industries: Sequence[str] | None = None,
    statuses: Sequence[str] | None = None, batch_years: Sequence[int] | None = None,
    min_team_size: int | None = None, min_score: float | None = None,
    query: str | None = None,
) -> pd.DataFrame:
    out = df
    if industries:
        out = out[out["industry"].isin(list(industries))]
    if statuses:
        out = out[out["status"].isin(list(statuses))]
    if batch_years:
        out = out[out["batch_year"].isin(list(batch_years))]
    if min_team_size is not None:
        out = out[out["team_size"].fillna(0) >= min_team_size]
    if min_score is not None:
        out = out[out["score"] >= min_score]
    if query and query.strip():
        needle = query.strip().lower()
        out = out[out.apply(lambda r: needle in _row_text(r), axis=1)]
    return out.reset_index(drop=True)


# ======================================================================= export
# Control chars Excel/openpyxl rejects (matches openpyxl's ILLEGAL_CHARACTERS_RE).
_ILLEGAL_XLSX_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")


def _clean_cell(value: object) -> object:
    if isinstance(value, str):
        return _ILLEGAL_XLSX_RE.sub("", value)
    return value


def _flatten_for_sheet(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        if out[col].apply(lambda v: isinstance(v, list)).any():
            out[col] = out[col].apply(
                lambda v: ", ".join(map(str, v)) if isinstance(v, list)
                else ("" if pd.isna(v) else v)
            )
    for col in out.columns:
        if out[col].dtype == object or pd.api.types.is_string_dtype(out[col]):
            out[col] = out[col].map(_clean_cell, na_action="ignore")
    return out


def _is_url_column(name: str) -> bool:
    return name == "website" or name.endswith("url")


def _style_workbook(xlsx_path: Path, columns: list[str]) -> None:
    from openpyxl import load_workbook

    wb = load_workbook(xlsx_path)
    ws = wb.active
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = f"A1:{get_column_letter(len(columns))}{ws.max_row}"
    url_cols = [i + 1 for i, name in enumerate(columns) if _is_url_column(name)]
    for col_idx in url_cols:
        for row in range(2, ws.max_row + 1):
            cell = ws.cell(row=row, column=col_idx)
            if isinstance(cell.value, str) and cell.value.startswith("http"):
                cell.hyperlink = cell.value
                cell.style = "Hyperlink"
    wb.save(xlsx_path)


def export(df: pd.DataFrame, *, out_dir: Path = Path("data/processed"),
           basename: str = "yc_radar") -> dict[str, Path]:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    parquet_path = out_dir / f"{basename}.parquet"
    csv_path = out_dir / f"{basename}.csv"
    xlsx_path = out_dir / f"{basename}.xlsx"
    df.to_parquet(parquet_path, index=False)
    flat = _flatten_for_sheet(df)
    flat.to_csv(csv_path, index=False)
    flat.to_excel(xlsx_path, index=False, engine="openpyxl")
    _style_workbook(xlsx_path, list(flat.columns))
    return {"parquet": parquet_path, "csv": csv_path, "xlsx": xlsx_path}


# ================================================================= orchestration
def build_dataset(*, force_refresh: bool = False, summarizer=None,
                  api_key: str | None = None) -> pd.DataFrame:
    """Run the whole pipeline and return the enriched, scored, annotated DataFrame.

    Pass ``summarizer=make_groq_summarizer(...)`` for free Groq AI summaries.
    """
    records = fetch_companies(force_refresh=force_refresh)
    df = normalize(records)
    df = add_investability(df)
    df = add_links(df)
    df = score(df)
    df = add_ai_summaries(df, api_key=api_key, summarizer=summarizer)
    df = merge_user_data(df)
    return df.sort_values("score", ascending=False).reset_index(drop=True)


## 3. Write the Streamlit dashboard (`app.py`)

In [ ]:
%%writefile app.py
"""Streamlit dashboard (Colab edition) — reads the exported Parquet."""
from pathlib import Path
import pandas as pd
import streamlit as st
import yc_radar_pipeline as yrp

DATASET = Path("data/processed/yc_radar.parquet")
USER_DATA = Path("data/user_data.csv")
LINK_COLUMNS = ["website", "yc_url", "news_url", "producthunt_url", "hn_url",
                "github_url", "wikipedia_url"]


@st.cache_data
def load_data(path: str) -> pd.DataFrame:
    return pd.read_parquet(path)


def main() -> None:
    st.set_page_config(page_title="YC Startup Radar", page_icon="\U0001F6F0\uFE0F", layout="wide")
    st.title("\U0001F6F0\uFE0F YC Startup Radar \u2014 2024\u20132026")
    if not DATASET.exists():
        st.warning("No dataset found. Run the pipeline cell in the notebook first.")
        st.stop()
    df = load_data(str(DATASET))
    df = yrp.merge_user_data(df, path=USER_DATA)

    st.sidebar.header("Filters")
    query = st.sidebar.text_input("Search (name / idea / tags)")
    industries = st.sidebar.multiselect("Industry", sorted(df["industry"].dropna().unique()))
    statuses = st.sidebar.multiselect("Status", sorted(df["status"].dropna().unique()))
    years = st.sidebar.multiselect("Batch year",
                                   sorted(int(y) for y in df["batch_year"].dropna().unique()))
    min_score = st.sidebar.slider("Minimum score", 0, 100, 0)
    max_team = int(df["team_size"].fillna(0).max() or 0)
    min_team = st.sidebar.slider("Minimum team size", 0, max_team, 0) if max_team else 0

    filtered = yrp.apply_filters(
        df, industries=industries or None, statuses=statuses or None,
        batch_years=years or None, min_team_size=min_team or None,
        min_score=min_score or None, query=query or None,
    ).sort_values("score", ascending=False)
    st.caption(f"Showing **{len(filtered)}** of {len(df)} companies")

    table_cols = [c for c in ["name", "batch", "industry", "status", "investability",
                              "team_size", "score", "one_liner", "website", "yc_url"]
                  if c in filtered.columns]
    col_config = {c: st.column_config.LinkColumn(c)
                  for c in ("website", "yc_url") if c in filtered.columns}
    st.dataframe(filtered[table_cols], use_container_width=True, hide_index=True,
                 column_config=col_config)

    st.subheader("Company details")
    for _, row in filtered.head(50).iterrows():
        with st.expander(f"{row['name']} \u2014 {row.get('one_liner', '')}  (score {row.get('score', '')})"):
            st.markdown(f"**Industry:** {row.get('industry', '')}  \n"
                        f"**Batch:** {row.get('batch', '')}  \n"
                        f"**Status:** {row.get('status', '')} \u2014 {row.get('investability', '')}  \n"
                        f"**Team size:** {row.get('team_size', '')}")
            if str(row.get("ai_summary", "")).strip():
                st.markdown(f"**AI summary:** {row['ai_summary']}")
            if str(row.get("ai_risk_notes", "")).strip():
                st.markdown(f"**Risks to check:** {row['ai_risk_notes']}")
            links = [f"[{c.replace('_url', '').replace('_', ' ').title() or 'Website'}]({row[c]})"
                     for c in LINK_COLUMNS if c in row and str(row.get(c, '')).startswith("http")]
            if links:
                st.markdown("**Open-source links:** " + " \u00b7 ".join(links))

    st.subheader("My shortlist & notes")
    st.caption("Edit rating (0\u20135), watchlist, notes, then Save (writes data/user_data.csv).")
    editor_cols = [c for c in ["slug", "name", "my_rating", "watchlist", "my_notes"]
                   if c in filtered.columns]
    edited = st.data_editor(
        filtered[editor_cols].copy(), use_container_width=True, hide_index=True,
        disabled=["slug", "name"],
        column_config={
            "my_rating": st.column_config.NumberColumn("Rating", min_value=0, max_value=5),
            "watchlist": st.column_config.CheckboxColumn("Watchlist"),
            "my_notes": st.column_config.TextColumn("Notes"),
        }, key="annotations_editor",
    )
    if st.button("\U0001F4BE Save annotations"):
        existing = yrp.load_user_data(USER_DATA).set_index("slug")
        for _, r in edited.iterrows():
            existing.loc[r["slug"]] = {"my_rating": r.get("my_rating"),
                                       "watchlist": bool(r.get("watchlist")),
                                       "my_notes": r.get("my_notes", "")}
        yrp.save_user_data(existing.reset_index(), path=USER_DATA)
        st.success(f"Saved annotations for {len(edited)} companies.")


if __name__ == "__main__":
    main()


## 4. (Optional) Claude API key for AI summaries — **paid, but cheap**

Get a key at **console.anthropic.com** (add a little credit). Put it in Colab **Secrets** (🔑 left sidebar) as `ANTHROPIC_API_KEY` (enable notebook access), or paste it in the prompt. The full YC 2024–2026 set costs roughly **3–4 USD** with Haiku 4.5 (short output + truncated input + resumable cache). Skip to run without AI.

In [ ]:
import os
try:
    from google.colab import userdata
    key = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    key = None
if not key:
    import getpass
    key = getpass.getpass('ANTHROPIC_API_KEY (blank to skip AI summaries): ').strip()
if key:
    os.environ['ANTHROPIC_API_KEY'] = key
    print('AI summaries: ENABLED via Claude Haiku 4.5 (paid, ~3-4 USD for the full set)')
else:
    print('AI summaries: disabled (placeholder text, no key, no cost)')

## 5. Run the pipeline → build the final Excel

Fetches live YC data, filters to 2024–2026, enriches, scores, and (if a Claude key is set) writes AI idea/risk summaries with **Haiku 4.5** — then exports `data/processed/yc_radar.xlsx` (+ `.parquet`, `.csv`). Progress prints as `X/N companies…`; the on-disk cache means re-runs only pay for **new** companies.

In [ ]:
import yc_radar_pipeline as yrp

claude_key = os.environ.get('ANTHROPIC_API_KEY')
summarizer = None
if claude_key:
    summarizer = yrp.make_claude_summarizer(
        claude_key,
        cache_path='data/processed/ai_cache.json',  # resume-safe: only new companies
        model='claude-haiku-4-5',   # cheapest capable model
        max_tokens=500,             # richer output for better review quality
        progress_every=50,          # prints 'X/N companies...'
    )

df = yrp.build_dataset(summarizer=summarizer)
paths = yrp.export(df)
print(f'{len(df)} companies -->', {k: str(v) for k, v in paths.items()})
df[['name', 'industry', 'status', 'investability', 'score', 'ai_summary']].head(15)

## 6. Download the Excel file

In [ ]:
from google.colab import files
files.download('data/processed/yc_radar.xlsx')
# To keep it in Google Drive instead:
#   from google.colab import drive; drive.mount('/content/drive')
#   import shutil; shutil.copy('data/processed/yc_radar.xlsx', '/content/drive/MyDrive/')

## 7. Launch the Streamlit dashboard (public link via cloudflared)

Starts the app and opens a **temporary public tunnel**. Click the `https://<random>.trycloudflare.com` URL that prints below. The link stays live while this cell runs — **stop the cell (■) to shut the dashboard down**.

In [ ]:
import os, subprocess, time, urllib.request

if not os.path.exists('cloudflared'):
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        'cloudflared')
    os.chmod('cloudflared', 0o755)

subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501',
                  '--server.headless', 'true', '--server.address', '0.0.0.0'])
time.sleep(6)
print('Opening public tunnel — click the trycloudflare.com link below:')
!./cloudflared tunnel --url http://localhost:8501 --no-autoupdate